# 波动率笔均量双因子多空策略

策略逻辑：基于GARCH波动率与笔均量因子构建多空组合

## 1. 导入依赖库

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# 设置中文显示
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False

## 2. 数据预处理

In [ ]:
# 读取个股数据
stock_file = r'data.csv'
df = pd.read_csv(stock_file)

# 读取沪深300基准数据
benchmark_file = r'沪深300.xlsx'
benchmark_df = pd.read_excel(benchmark_file)

# 数据清洗
df['date'] = pd.to_datetime(df['date'])
benchmark_df['时间'] = pd.to_datetime(benchmark_df['时间'])
benchmark_df.columns = ['date', 'close']

# 剔除缺失值
df = df.dropna(subset=['close', 'prev_close', 'volume', 'num_trades'])

# 剔除异常值（价格非正）
df = df[(df['close'] > 0) & (df['prev_close'] > 0)]

# 剔除涨跌停样本（当日无法交易）
df = df[(df['close'] < df['limit_up']) & (df['close'] > df['limit_down'])]

# 计算日对数收益率
df['log_return'] = np.log(df['close'] / df['prev_close'])

# 计算沪深300对数收益率
benchmark_df = benchmark_df.sort_values('date').reset_index(drop=True)
benchmark_df['log_return'] = np.log(benchmark_df['close'] / benchmark_df['close'].shift(1))
benchmark_df = benchmark_df.dropna()

print(f"数据预处理完成，共 {df['asset'].nunique()} 只股票，{df['date'].nunique()} 个交易日")
print(f"时间范围: {df['date'].min()} 至 {df['date'].max()}")

## 3. 三类波动率计算

In [ ]:
# 按股票分组计算
def calc_volatility_metrics(df):
    """计算三种波动率指标"""
    df = df.sort_values('date').copy()
    
    # 1. 5日滚动波动率
    df['rolling_vol_5'] = df['log_return'].rolling(window=5, min_periods=3).std() * np.sqrt(252)
    
    # 2. EWMA波动率 (λ=0.94)
    lambda_ewma = 0.94
    df['ewma_vol'] = df['log_return'].ewm(alpha=1-lambda_ewma, min_periods=10).std() * np.sqrt(252)
    
    return df

# 应用计算
df = df.groupby('asset', group_keys=False).apply(calc_volatility_metrics)

print("滚动波动率和EWMA波动率计算完成")

## 4. GARCH(1,1)波动率计算（MLE估计）

In [ ]:
def garch11_mle(returns):
    """使用MLE估计GARCH(1,1)参数"""
    returns = returns.dropna().values
    if len(returns) < 30:
        return None, None, None, None
    
    T = len(returns)
    
    # 对数似然函数
    def log_likelihood(params):
        omega, alpha, beta = params
        if omega <= 0 or alpha < 0 or beta < 0 or alpha + beta >= 1:
            return 1e10
        
        sigma2 = np.zeros(T)
        sigma2[0] = np.var(returns)
        
        for t in range(1, T):
            sigma2[t] = omega + alpha * returns[t-1]**2 + beta * sigma2[t-1]
        
        ll = -0.5 * np.sum(np.log(2 * np.pi * sigma2) + returns**2 / sigma2)
        return -ll
    
    # 初始参数
    x0 = [0.000001, 0.1, 0.85]
    bounds = [(1e-8, None), (0, 0.999), (0, 0.999)]
    
    try:
        result = minimize(log_likelihood, x0, bounds=bounds, method='L-BFGS-B')
        if result.success:
            omega, alpha, beta = result.x
            long_run_var = omega / (1 - alpha - beta) if (1 - alpha - beta) > 0 else np.var(returns)
            return omega, alpha, beta, long_run_var
    except:
        pass
    
    return None, None, None, None

def calc_garch_vol(df):
    """计算GARCH(1,1)动态波动率"""
    df = df.sort_values('date').copy()
    returns = df['log_return'].dropna()
    
    omega, alpha, beta, long_run_var = garch11_mle(returns)
    
    if omega is None:
        df['garch_vol'] = df['log_return'].rolling(window=20, min_periods=10).std() * np.sqrt(252)
        df['garch_omega'] = np.nan
        df['garch_alpha'] = np.nan
        df['garch_beta'] = np.nan
        df['garch_long_run_var'] = np.nan
        return df
    
    # 计算条件方差序列
    T = len(df)
    sigma2 = np.zeros(T)
    sigma2[0] = long_run_var if long_run_var > 0 else np.var(returns)
    
    for t in range(1, T):
        ret_t = df['log_return'].iloc[t-1]
        if not np.isnan(ret_t):
            sigma2[t] = omega + alpha * ret_t**2 + beta * sigma2[t-1]
        else:
            sigma2[t] = sigma2[t-1]
    
    df['garch_vol'] = np.sqrt(sigma2) * np.sqrt(252)
    df['garch_omega'] = omega
    df['garch_alpha'] = alpha
    df['garch_beta'] = beta
    df['garch_long_run_var'] = long_run_var
    
    return df

# 计算GARCH波动率
df = df.groupby('asset', group_keys=False).apply(calc_garch_vol)

# 输出GARCH参数示例
sample_stock = df['asset'].iloc[0]
sample_params = df[df['asset'] == sample_stock][['garch_omega', 'garch_alpha', 'garch_beta', 'garch_long_run_var']].iloc[0]
print(f"\nGARCH(1,1)参数示例 (股票 {sample_stock}):")
print(f"  ω (omega): {sample_params['garch_omega']:.8f}")
print(f"  α (alpha): {sample_params['garch_alpha']:.4f}")
print(f"  β (beta): {sample_params['garch_beta']:.4f}")
print(f"  长期无条件方差: {sample_params['garch_long_run_var']:.8f}")
print(f"  长期波动率: {np.sqrt(sample_params['garch_long_run_var']) * np.sqrt(252):.4f}")

## 5. 波动率对比图

In [ ]:
# 选取一只股票展示三种波动率对比
sample_stock = df['asset'].iloc[0]
sample_df = df[df['asset'] == sample_stock].copy()

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(sample_df['date'], sample_df['rolling_vol_5'], label='5日滚动波动率', alpha=0.7)
ax.plot(sample_df['date'], sample_df['ewma_vol'], label='EWMA波动率(λ=0.94)', alpha=0.7)
ax.plot(sample_df['date'], sample_df['garch_vol'], label='GARCH(1,1)波动率', alpha=0.9)
ax.set_xlabel('日期')
ax.set_ylabel('年化波动率')
ax.set_title(f'股票 {sample_stock} 三类波动率对比')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("波动率对比图展示完成")

## 6. 笔均量因子构建

In [ ]:
# 计算笔均量因子 = 成交量 / 成交笔数
df['volume_per_trade'] = df['volume'] / df['num_trades']

# 去极值函数（MAD法）
def winsorize_mad(x, n=3):
    """MAD去极值"""
    median = x.median()
    mad = np.median(np.abs(x - median))
    upper = median + n * 1.4826 * mad
    lower = median - n * 1.4826 * mad
    return x.clip(lower, upper)

# Z-score标准化函数
def zscore_standardize(x):
    """截面Z-score标准化"""
    mean = x.mean()
    std = x.std()
    if std == 0 or pd.isna(std):
        return x - mean
    return (x - mean) / std

# 每日截面处理
def process_factor_daily(group):
    """每日因子处理：去极值 + Z-score标准化"""
    group = group.copy()
    
    # 笔均量因子处理
    group['volume_per_trade'] = winsorize_mad(group['volume_per_trade'])
    group['volume_per_trade_z'] = zscore_standardize(group['volume_per_trade'])
    
    # GARCH波动率因子处理
    group['garch_vol'] = winsorize_mad(group['garch_vol'])
    group['garch_vol_z'] = zscore_standardize(group['garch_vol'])
    
    return group

# 应用每日截面处理
df = df.groupby('date', group_keys=False).apply(process_factor_daily)

# 剔除处理后的缺失值
df = df.dropna(subset=['garch_vol_z', 'volume_per_trade_z', 'log_return'])

print(f"笔均量因子构建完成，有效样本数: {len(df)}")

## 7. 因子有效性检验（IC、IR）

In [ ]:
# 计算未来一期收益率
df = df.sort_values(['asset', 'date']).copy()
df['next_return'] = df.groupby('asset')['log_return'].shift(-1)

# 计算每日IC
def calc_daily_ic(group):
    """计算单日IC"""
    group = group.dropna(subset=['next_return'])
    if len(group) < 5:
        return pd.Series({'ic_garch': np.nan, 'ic_volume': np.nan})
    
    ic_garch = stats.spearmanr(group['garch_vol_z'], group['next_return'])[0]
    ic_volume = stats.spearmanr(group['volume_per_trade_z'], group['next_return'])[0]
    
    return pd.Series({'ic_garch': ic_garch, 'ic_volume': ic_volume})

daily_ic = df.groupby('date').apply(calc_daily_ic, include_groups=False)
daily_ic = daily_ic.dropna()

# 计算IC统计指标
ic_stats = pd.DataFrame({
    'GARCH波动率因子': [
        daily_ic['ic_garch'].mean(),
        daily_ic['ic_garch'].std(),
        daily_ic['ic_garch'].mean() / daily_ic['ic_garch'].std() if daily_ic['ic_garch'].std() > 0 else np.nan,
        (daily_ic['ic_garch'] > 0).sum() / len(daily_ic)
    ],
    '笔均量因子': [
        daily_ic['ic_volume'].mean(),
        daily_ic['ic_volume'].std(),
        daily_ic['ic_volume'].mean() / daily_ic['ic_volume'].std() if daily_ic['ic_volume'].std() > 0 else np.nan,
        (daily_ic['ic_volume'] > 0).sum() / len(daily_ic)
    ]
}, index=['IC均值', 'IC标准差', 'IR比率', 'IC>0占比'])

print("\n因子IC/IR统计结果:")
print(ic_stats.round(4))

## 8. 5分组回测（验证收益单调性）

In [ ]:
def quintile_backtest(df, factor_col, n_groups=5):
    """五分组回测"""
    df = df.copy()
    
    # 每日分组
    def assign_quintile(group):
        group = group.copy()
        group['quintile'] = pd.qcut(group[factor_col], q=n_groups, labels=range(1, n_groups+1), duplicates='drop')
        return group
    
    df = df.groupby('date', group_keys=False).apply(assign_quintile)
    df = df.dropna(subset=['quintile', 'next_return'])
    
    # 计算每组每日收益
    daily_returns = df.groupby(['date', 'quintile'])['next_return'].mean().reset_index()
    daily_returns = daily_returns.pivot(index='date', columns='quintile', values='next_return')
    
    # 计算累计净值
    cumulative_returns = (1 + daily_returns.fillna(0)).cumprod()
    
    # 计算年化收益
    annual_returns = daily_returns.mean() * 252
    
    return daily_returns, cumulative_returns, annual_returns

# GARCH波动率因子分组回测
daily_ret_garch, cum_ret_garch, ann_ret_garch = quintile_backtest(df, 'garch_vol_z')

# 笔均量因子分组回测
daily_ret_volume, cum_ret_volume, ann_ret_volume = quintile_backtest(df, 'volume_per_trade_z')

print("\nGARCH波动率因子5分组年化收益:")
for i in range(1, 6):
    if i in ann_ret_garch.index:
        print(f"  第{i}组: {ann_ret_garch[i]*100:.2f}%")

print("\n笔均量因子5分组年化收益:")
for i in range(1, 6):
    if i in ann_ret_volume.index:
        print(f"  第{i}组: {ann_ret_volume[i]*100:.2f}%")

## 9. 分组收益图

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# GARCH波动率因子分组净值
for col in cum_ret_garch.columns:
    axes[0].plot(cum_ret_garch.index, cum_ret_garch[col], label=f'第{int(col)}组')
axes[0].set_title('GARCH波动率因子5分组累计净值')
axes[0].set_xlabel('日期')
axes[0].set_ylabel('累计净值')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 笔均量因子分组净值
for col in cum_ret_volume.columns:
    axes[1].plot(cum_ret_volume.index, cum_ret_volume[col], label=f'第{int(col)}组')
axes[1].set_title('笔均量因子5分组累计净值')
axes[1].set_xlabel('日期')
axes[1].set_ylabel('累计净值')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("分组收益图展示完成")

## 10. 多空策略回测

In [ ]:
def long_short_backtest(df, factor_col, long_pct=0.1, short_pct=0.1, transaction_cost=0.001):
    """多空策略回测"""
    df = df.copy()
    
    # 每日选股
    def select_stocks(group):
        group = group.copy()
        n = len(group)
        if n < 10:
            return group
        
        n_long = max(1, int(n * long_pct))
        n_short = max(1, int(n * short_pct))
        
        group_sorted = group.sort_values(factor_col)
        group['position'] = 0
        group.loc[group_sorted.index[-n_long:], 'position'] = 1  # 做多
        group.loc[group_sorted.index[:n_short], 'position'] = -1  # 做空
        
        return group
    
    df = df.groupby('date', group_keys=False).apply(select_stocks)
    df = df[df['position'] != 0].copy()
    
    # 计算每日收益
    daily_pnl = df.groupby('date').apply(
        lambda x: (x['position'] * x['next_return']).sum() / x['position'].abs().sum()
    )
    
    # 扣除交易成本
    daily_pnl = daily_pnl - transaction_cost * 2  # 双边成本
    
    # 计算累计净值
    nav = (1 + daily_pnl.fillna(0)).cumprod()
    
    return daily_pnl, nav

def dual_factor_backtest(df, long_pct=0.2, short_pct=0.2, transaction_cost=0.001):
    """双因子多空策略"""
    df = df.copy()
    
    # 计算综合得分（两个因子同时排名前20%或后20%）
    def select_dual_factor(group):
        group = group.copy()
        n = len(group)
        if n < 10:
            return group
        
        # 计算排名百分比
        group['garch_rank'] = group['garch_vol_z'].rank(pct=True)
        group['volume_rank'] = group['volume_per_trade_z'].rank(pct=True)
        
        # 双因子同时前20%做多，同时后20%做空
        group['position'] = 0
        group.loc[(group['garch_rank'] >= 0.8) & (group['volume_rank'] >= 0.8), 'position'] = 1
        group.loc[(group['garch_rank'] <= 0.2) & (group['volume_rank'] <= 0.2), 'position'] = -1
        
        return group
    
    df = df.groupby('date', group_keys=False).apply(select_dual_factor)
    df = df[df['position'] != 0].copy()
    
    # 计算每日收益
    daily_pnl = df.groupby('date').apply(
        lambda x: (x['position'] * x['next_return']).sum() / x['position'].abs().sum()
    )
    
    # 扣除交易成本
    daily_pnl = daily_pnl - transaction_cost * 2
    
    # 计算累计净值
    nav = (1 + daily_pnl.fillna(0)).cumprod()
    
    return daily_pnl, nav

# 单因子策略
daily_pnl_garch_ls, nav_garch_ls = long_short_backtest(df, 'garch_vol_z', long_pct=0.1, short_pct=0.1)

# 双因子策略
daily_pnl_dual, nav_dual = dual_factor_backtest(df, long_pct=0.2, short_pct=0.2)

print("多空策略回测完成")

## 11. 基准对比

In [ ]:
# 对齐日期
common_dates = nav_garch_ls.index.intersection(benchmark_df['date'])

# 截取共同时间段
nav_garch_aligned = nav_garch_ls[nav_garch_ls.index.isin(common_dates)]
nav_dual_aligned = nav_dual[nav_dual.index.isin(common_dates)]
benchmark_aligned = benchmark_df[benchmark_df['date'].isin(common_dates)].set_index('date')

# 计算基准净值
benchmark_nav = (1 + benchmark_aligned['log_return'].fillna(0)).cumprod()

# 确保日期对齐
all_dates = nav_garch_aligned.index.union(nav_dual_aligned.index).union(benchmark_nav.index)
nav_garch_aligned = nav_garch_aligned.reindex(all_dates, method='ffill')
nav_dual_aligned = nav_dual_aligned.reindex(all_dates, method='ffill')
benchmark_nav = benchmark_nav.reindex(all_dates, method='ffill')

print(f"\n回测时间段: {all_dates.min()} 至 {all_dates.max()}")
print(f"总交易日: {len(all_dates)}")

## 12. 策略绩效指标计算

In [ ]:
def calc_performance_metrics(daily_returns, nav):
    """计算策略绩效指标"""
    daily_returns = daily_returns.dropna()
    
    if len(daily_returns) == 0:
        return pd.Series([np.nan]*7, index=['累计净值', '年化收益', '年化波动', '夏普比率', '最大回撤', '胜率', '换手率'])
    
    # 累计净值
    final_nav = nav.iloc[-1] if len(nav) > 0 else 1
    
    # 年化收益
    annual_return = daily_returns.mean() * 252
    
    # 年化波动
    annual_vol = daily_returns.std() * np.sqrt(252)
    
    # 夏普比率（假设无风险利率为0）
    sharpe = annual_return / annual_vol if annual_vol > 0 else np.nan
    
    # 最大回撤
    cummax = nav.cummax()
    drawdown = (nav - cummax) / cummax
    max_drawdown = drawdown.min()
    
    # 胜率
    win_rate = (daily_returns > 0).sum() / len(daily_returns)
    
    # 换手率（近似估算）
    turnover = 1.0  # 日频调仓，近似每日100%换手
    
    return pd.Series({
        '累计净值': final_nav,
        '年化收益': annual_return,
        '年化波动': annual_vol,
        '夏普比率': sharpe,
        '最大回撤': max_drawdown,
        '胜率': win_rate,
        '换手率': turnover
    })

# 计算各策略绩效
performance = pd.DataFrame({
    'GARCH单因子多空': calc_performance_metrics(daily_pnl_garch_ls.reindex(all_dates), nav_garch_aligned),
    '双因子多空': calc_performance_metrics(daily_pnl_dual.reindex(all_dates), nav_dual_aligned),
    '沪深300基准': calc_performance_metrics(benchmark_aligned['log_return'].reindex(all_dates), benchmark_nav)
})

print("\n策略绩效对比:")
print(performance.round(4))

## 13. 净值对比图

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

ax.plot(nav_garch_aligned.index, nav_garch_aligned, label='GARCH单因子多空', linewidth=1.5)
ax.plot(nav_dual_aligned.index, nav_dual_aligned, label='双因子多空', linewidth=1.5)
ax.plot(benchmark_nav.index, benchmark_nav, label='沪深300基准', linewidth=1.5, alpha=0.7)

ax.set_xlabel('日期')
ax.set_ylabel('累计净值')
ax.set_title('策略净值对比')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("净值对比图展示完成")

## 14. 回撤图

In [ ]:
# 计算回撤
def calc_drawdown(nav):
    cummax = nav.cummax()
    return (nav - cummax) / cummax

dd_garch = calc_drawdown(nav_garch_aligned)
dd_dual = calc_drawdown(nav_dual_aligned)
dd_benchmark = calc_drawdown(benchmark_nav)

fig, ax = plt.subplots(figsize=(12, 6))

ax.fill_between(dd_garch.index, dd_garch, 0, alpha=0.3, label='GARCH单因子多空')
ax.fill_between(dd_dual.index, dd_dual, 0, alpha=0.3, label='双因子多空')
ax.fill_between(dd_benchmark.index, dd_benchmark, 0, alpha=0.3, label='沪深300基准')

ax.set_xlabel('日期')
ax.set_ylabel('回撤')
ax.set_title('策略回撤对比')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("回撤图展示完成")

## 15. 结果分析

In [ ]:
sep = chr(61) * 60
print(sep)
print("策略回测结果总结")
print(sep)

print("\n【数据概况】")
print(f"回测期间: {df['date'].min().strftime('%Y-%m-%d')} 至 {df['date'].max().strftime('%Y-%m-%d')}")
print(f"股票数量: {df['asset'].nunique()} 只")
print(f"交易日数: {df['date'].nunique()} 天")

print("\n【GARCH(1,1)模型参数】")
valid_params = df.dropna(subset=['garch_omega'])[['asset', 'garch_omega', 'garch_alpha', 'garch_beta', 'garch_long_run_var']].drop_duplicates('asset')
if len(valid_params) > 0:
    print(f"有效估计股票数: {len(valid_params)}")
    print(f"α均值: {valid_params['garch_alpha'].mean():.4f}")
    print(f"β均值: {valid_params['garch_beta'].mean():.4f}")
    print(f"α+β均值: {(valid_params['garch_alpha'] + valid_params['garch_beta']).mean():.4f}")

print("\n【因子有效性】")
print(f"GARCH波动率因子 IC均值: {ic_stats.loc['IC均值', 'GARCH波动率因子']:.4f}")
print(f"GARCH波动率因子 IR比率: {ic_stats.loc['IR比率', 'GARCH波动率因子']:.4f}")
print(f"笔均量因子 IC均值: {ic_stats.loc['IC均值', '笔均量因子']:.4f}")
print(f"笔均量因子 IR比率: {ic_stats.loc['IR比率', '笔均量因子']:.4f}")

print("\n【策略绩效】")
for col in performance.columns:
    print(f"\n{col}:")
    print(f"  累计净值: {performance.loc['累计净值', col]:.4f}")
    print(f"  年化收益: {performance.loc['年化收益', col]*100:.2f}%")
    print(f"  夏普比率: {performance.loc['夏普比率', col]:.4f}")
    print(f"  最大回撤: {performance.loc['最大回撤', col]*100:.2f}%")

print("\n【策略逻辑说明】")
print("1. GARCH单因子多空: 每日选取GARCH波动率前10%做多，后10%做空")
print("2. 双因子多空: 每日选取GARCH波动率和笔均量同时前20%做多，同时后20%做空")
print("3. 交易成本: 双边0.1%")
print("4. 调仓频率: 日频")
print(sep)